In [11]:
import numpy as np
import pandas as pd

Max_Depth = 3
Folds = 5

data = pd.read_csv("/Users/shivam/Downloads/yeast.csv")
features = ["mcg", "gvh", "alm", "mit", "erl", "pox", "vac", "nuc"]
X = data[features].values
labels = data["name"].values.astype(str)

In [12]:
def entropy(group):
    names, counts = np.unique(group, return_counts=True)
    p = counts / len(group)
    return -np.sum(p * np.log2(p))

In [13]:
def information_gain(parent, yes, no):
    weight_yes = len(yes) / len(parent)
    weight_no = len(no) / len(parent)
    entropy_after = weight_yes * entropy(yes) + weight_no * entropy(no)
    return entropy(parent) - entropy_after

In [14]:
def best_cut(values, group):
    best_gain = 0
    best_value = None
    for cut in np.unique(values):
        yes = group[values <= cut]
        no = group[values > cut]
        gain = information_gain(group, yes, no)
        if gain > best_gain:
            best_gain = gain
            best_value = cut
    return best_gain, best_value

In [15]:
def build_tree(X, group, depth):
    most_common = pd.Series(group).value_counts().idxmax()
    leaf = {"answer": most_common, "rows": len(group)}

    if depth == Max_Depth or entropy(group) == 0:
        return leaf

    best_gain = 0
    best_column = None
    best_value = None
    for column in range(len(features)):
        gain, cut = best_cut(X[:, column], group)
        if gain > best_gain:
            best_gain = gain
            best_column = column
            best_value = cut

    if best_column is None:
        return leaf

    yes_rows = X[:, best_column] <= best_value
    no_rows = X[:, best_column] > best_value
    return {
        "column": best_column,
        "cut": best_value,
        "gain": best_gain,
        "yes": build_tree(X[yes_rows], group[yes_rows], depth + 1),
        "no": build_tree(X[no_rows], group[no_rows], depth + 1)
    }

In [16]:
def predict(tree, row):
    while "column" in tree:
        if row[tree["column"]] <= tree["cut"]:
            tree = tree["yes"]
        else:
            tree = tree["no"]
    return tree["answer"]

In [17]:
def f1_score(true, predicted):
    scores = []
    for name in np.unique(true):
        tp = 0
        fp = 0
        fn = 0
        for i in range(len(true)):
            if predicted[i] == name and true[i] == name:
                tp = tp + 1
            elif predicted[i] == name and true[i] != name:
                fp = fp + 1
            elif predicted[i] != name and true[i] == name:
                fn = fn + 1
        if tp == 0:
            f1 = 0
        else:
            precision = tp / (tp + fp)
            recall = tp / (tp + fn)
            f1 = 2 * precision * recall / (precision + recall)
        scores.append(f1)
    return np.mean(scores)

In [18]:
print("Information gain of each column at the root:")
for column in range(len(features)):
    gain, cut = best_cut(X[:, column], labels)
    print("   ", features[column], " gain =", round(gain, 3), " cut-off =", cut)

tree = build_tree(X, labels, 0)
print("\nRoot question:", features[tree["column"]], "<=", tree["cut"])

Information gain of each column at the root:
    mcg  gain = 0.272  cut-off = 0.67
    gvh  gain = 0.189  cut-off = 0.65
    alm  gain = 0.376  cut-off = 0.43
    mit  gain = 0.138  cut-off = 0.32
    erl  gain = 0.027  cut-off = 0.5
    pox  gain = 0.045  cut-off = 0.0
    vac  gain = 0.026  cut-off = 0.5
    nuc  gain = 0.087  cut-off = 0.24

Root question: alm <= 0.43


In [20]:
np.random.seed(0)
fold_of_row = np.zeros(len(labels))
for name in np.unique(labels):
    rows_of_this_type = np.where(labels == name)[0]
    np.random.shuffle(rows_of_this_type)
    for position in range(len(rows_of_this_type)):
        row = rows_of_this_type[position]
        fold_of_row[row] = position % Folds

print("\nCross-validation :")
all_scores = []
for fold in range(FOLDS):
    test_mask = (fold_of_row == fold)
    train_mask = (fold_of_row != fold)
    
    X_train = X[train_mask]
    y_train = labels[train_mask]
    X_test = X[test_mask]
    y_test = labels[test_mask]
    
    fold_tree = build_tree(X_train, y_train, 0)
    
    predictions = []
    for row in X_test:
        pred = predict(fold_tree, row)
        predictions.append(pred)
        
    score = f1_score(y_test, predictions)
    all_scores.append(score)
    print("  Fold", fold + 1, " F1 =", round(score, 3))

print("Average F1 =", round(np.mean(all_scores), 3))


Cross-validation :
  Fold 1  F1 = 0.379
  Fold 2  F1 = 0.361
  Fold 3  F1 = 0.386
  Fold 4  F1 = 0.321
  Fold 5  F1 = 0.338
Average F1 = 0.357
